# Tanzania VACS 2024 — PUD exploration

Single public file: **`TANZANIA_VACS_2024_PUD.dta`** (`data/raw/Tanzania 2024/`). Males and females are **in one dataset** (split-sample EAs; use **`SEX`**).

**Flow:** (1) Load → (2) **§2 — column list & quick EDA** → **checklist** (`utils.checklist`, **`type_and_width`** + TSV) → (3) **§3 — further EDA** (row samples, derived HH fields, slot summaries, duplicates) → (4) **§4 — harmonized codebook**.

**Reusing for other countries:** duplicate this notebook, change `COUNTRY_*` paths and the mapping dict in the harmonized markdown if variable names differ.

In [ ]:
from pathlib import Path
import sys

from IPython.display import display

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyreadstat
import re

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# --- Tanzania 2024 (edit for another country) ---
COUNTRY_DIR = ROOT / "data" / "raw" / "Tanzania 2024"
PUD_NAME = "TANZANIA_VACS_2024_PUD.dta"
DTA_PATH = COUNTRY_DIR / PUD_NAME

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("ggplot")

## 1. Load data

`pyreadstat.read_dta` returns `df` and `meta` (`column_names_to_labels`, value labels when present).

In [ ]:
if not DTA_PATH.is_file():
    raise FileNotFoundError(f"Expected:\n  {DTA_PATH}")

df, meta = pyreadstat.read_dta(DTA_PATH)
print(f"File: {DTA_PATH}")
print(f"Rows × columns: {df.shape[0]:,} × {df.shape[1]:,}")
if getattr(meta, "file_label", None):
    print(f"Stata dataset label: {meta.file_label!r}")
if "SEX" in df.columns:
    print("SEX counts:")
    display(df["SEX"].value_counts(dropna=False))
df.head()

## 2. Column list & quick EDA

Stata labels, dtypes, and missingness: full `var_table` preview (first 40 columns), the 15 columns with highest missingness, and `df.info`. Then run the **checklist** cells (**`utils.checklist`**); edit **`CANDIDATES`** if needed.


In [ ]:
name_to_label = dict(meta.column_names_to_labels) if meta.column_names_to_labels else {}
var_table = pd.DataFrame({
    "column": df.columns,
    "stata_label": [name_to_label.get(c, "") or "" for c in df.columns],
    "dtype": df.dtypes.astype(str).values,
    "missing_n": df.isna().sum().values,
    "missing_pct": (100 * df.isna().mean()).round(2),
})

print(f"Variables: {len(df.columns):,}  |  Observations: {len(df):,}")
print(f"Embedded Stata value-label maps: {len(meta.value_labels or {})}")
display(var_table.head(40))
display(
    var_table.sort_values("missing_pct", ascending=False)
    .head(15)
    .reset_index(drop=True)
)
df.info(max_cols=20)

### Harmonized geography / ID checklist (`utils.checklist`)

**Admin 2 = enumeration areas (EAs):** **`CLUSTER`** (hyphenated **`NNN-NNN`** string in this extract) is the EA / cluster identifier for this project—not a substitute with **`PSU`** without checking the User Guide (**`PSU`** is numeric design).

**Geo:** **`REGION`** (coarsest), **`AREA`** (urban/rural-type layer). Edit **`CANDIDATES`** after §2 if you add columns.

See **`skills/memory.md`** for PI width / layout conventions.


In [ ]:
# You choose candidates after §2 EDA; utils summarize columns present in `df`.
from utils.checklist import build_checklist_df, checklist_to_tsv

CANDIDATES = [
    ("Admin 1 (region)", ["REGION"]),
    ("Area (urban/rural etc.)", ["AREA"]),
    ("Admin 2 — enumeration area (EA)", ["CLUSTER"]),
    ("PSU (numeric design)", ["PSU"]),
    ("Stratum", ["STRATA"]),
    ("Respondent ID", ["PUD_ID"]),
    ("Roster / HH context", ["NTOT", "REGI_C"]),
    ("Sex", ["SEX"]),
    ("Weights", ["SAMPLEWEIGHT", "HIVWEIGHT"]),
]

_labels = meta.column_names_to_labels or {}
checklist_df = build_checklist_df(df, CANDIDATES, column_labels=_labels)

with pd.option_context("display.max_colwidth", 100, "display.width", 220):
    display(checklist_df)

print("\n--- TSV (copy for Excel / codebook) ---\n")
print(checklist_to_tsv(checklist_df))


## 3. Further EDA and exploration

### Raw row samples

Full width is large; below: **head / tail / sample** plus a **core** slice for IDs, geography, cluster, and weights.


In [ ]:
pd.set_option("display.max_columns", 35)
pd.set_option("display.width", 120)
pd.set_option("display.max_colwidth", 60)

print("--- head(6) ---")
display(df.head(6))
print("\n--- tail(3) ---")
display(df.tail(3))
print("\n--- sample(5, random_state=0) ---")
display(df.sample(5, random_state=0))

_core_cols = [
    c
    for c in [
        "PUD_ID",
        "PSU",
        "CLUSTER",
        "REGION",
        "REGI_C",
        "AREA",
        "STRATA",
        "SEX",
        "AGE",
        "NTOT",
        "SAMPLEWEIGHT",
        "HIVWEIGHT",
    ]
    if c in df.columns
]
subset = df[_core_cols]
print(f"\n--- core columns ({len(_core_cols)}) ---")
display(subset.head(8))
display(subset.sample(5, random_state=1))

### Household line, slot summaries, and duplicates

Derives **`HH_LINE`** / **`HH_KEY`**, prints per-slot summaries aligned with the harmonized slots, and lists duplicate **`PUD_ID`** rows.


In [ ]:
import re

L = meta.column_names_to_labels or {}

df = df.copy()
df["HH_LINE"] = df["PUD_ID"].astype(str).str.split("_").str[-1]
df["HH_KEY"] = df["CLUSTER"].astype(str) + "_" + df["HH_LINE"]

# --- Per-variable summary format (compact) ---
# (1) Width: **character length** (min–max) for string/object columns — not “digit count” through the string.
#     For **numeric** columns only: digit count from integer string form (e.g. PSU codes) + min/max.
# (2) Optional **style** template when all values share one pattern (e.g. NNN-NNN for cluster IDs).
# (3) Male/Female text heuristic (not single-letter M/F codes) on string values.
# (4) dtype; n_distinct; missing; Stata label.

_WORD_SEX = re.compile(r"(?:male|females?|female)", re.IGNORECASE)
# Underscore-delimited segments like Female_… in long IDs (not single-letter TLQC/IQC codes "M", "F")
_EMBED_MF = re.compile(r"(?i)(?:^|_)(?:male|female)(?=_|$)")

def _abstract_digit_pattern(val: str) -> str:
    """Digit runs -> NNN; letter runs -> A; other chars literal (e.g. 133-133 -> NNN-NNN)."""
    parts = []
    i = 0
    while i < len(val):
        ch = val[i]
        if ch.isdigit():
            j = i
            while j < len(val) and val[j].isdigit():
                j += 1
            parts.append("N" * (j - i))
            i = j
        elif ch.isalpha():
            j = i
            while j < len(val) and val[j].isalpha():
                j += 1
            parts.append("A")
            i = j
        else:
            parts.append(ch)
            i += 1
    return "".join(parts)


def _unified_style_pattern(st: pd.Series):
    st = st.dropna().astype(str)
    if len(st) == 0:
        return None
    abstracts = st.map(_abstract_digit_pattern)
    if abstracts.nunique(dropna=False) != 1:
        return None
    pat = abstracts.iloc[0]
    if not any(ch.isdigit() for ch in pat):
        return None
    if set(pat) <= {"N"}:
        return None
    ex = st.iloc[0]
    if len(pat) > 72:
        return f"{pat[:72]}… (e.g. {ex[:40]}{'…' if len(ex) > 40 else ''})"
    return f"{pat} (e.g. {ex})"


def _width_note(s: pd.Series) -> str:
    sn = s.dropna()
    if len(sn) == 0:
        return "n/a"
    if pd.api.types.is_numeric_dtype(s):
        whole = (sn == sn.astype(float).astype(int)).all()
        if whole:
            lens = sn.astype(int).astype(str).str.len()
            lo, hi = int(lens.min()), int(lens.max())
            return f"{lo}-{hi} digits (integer codes)" if lo != hi else f"{lo} digits (integer codes)"
        lens = sn.astype(str).str.len()
        lo, hi = int(lens.min()), int(lens.max())
        return f"{lo}-{hi} chars (numeric as string)" if lo != hi else f"{lo} chars (numeric as string)"
    st = sn.astype(str)
    lens = st.str.len()
    lo, hi = int(lens.min()), int(lens.max())
    w = f"{lo}-{hi} chars" if lo != hi else f"{lo} chars"
    if st.str.fullmatch(r"\d+").all():
        return f"{w} (string; all numeric characters)"
    return f"{w} (string)"


def _special_id_note(s: pd.Series) -> str:
    if pd.api.types.is_numeric_dtype(s):
        return "no M/F identifier (numeric)"
    st = s.dropna().astype(str)
    if len(st) == 0:
        return "n/a"
    if st.str.contains(_WORD_SEX, regex=True, na=False).any() or st.str.contains(_EMBED_MF, regex=True, na=False).any():
        return "Male/Female text (words or _Female_/_Male_ segments)"
    return "no male/female text (heuristic)"


def slot_summary(title, cols, note_extra=""):
    """Print one slot: width (chars for text, digit-width note for numeric), optional style, dtype, counts."""
    print("\n" + "=" * 60)
    print(title)
    print("=" * 60)
    miss = [c for c in cols if c not in df.columns]
    if miss:
        print("MISSING columns:", miss)
        return
    for c in cols:
        s = df[c]
        lbl = (L.get(c) or "")[:75]
        size_part = _width_note(s)
        style = _unified_style_pattern(s) if not pd.api.types.is_numeric_dtype(s) else None
        style_part = f"; style {style}" if style else ""
        id_part = _special_id_note(s)
        if pd.api.types.is_numeric_dtype(s):
            sn = s.dropna()
            extra = f"min/max={sn.min()}/{sn.max()}" if len(sn) else "min/max=n/a"
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}; {extra}"
        else:
            info = f"{size_part}{style_part}; {id_part}; dtype={s.dtype}"
        print(f"  {c} | {lbl}")
        print(f"    {info}; n_distinct={s.nunique(dropna=True)}; missing={s.isna().sum()}")
    if note_extra:
        print("  ", note_extra)


slot_summary("1. Respondent ID", ["PUD_ID"], "unique rows: " + str(df["PUD_ID"].nunique()) + f" / {len(df)}")
slot_summary("2. Household pieces", ["CLUSTER", "HH_LINE", "HH_KEY"], "dup HH_KEY: " + str(int(df["HH_KEY"].duplicated().sum())))
slot_summary("3. Geo level 1", ["REGION"])
slot_summary("4. Geo level 2 / urb-rural", ["AREA", "REGI_C"])
slot_summary("5. Cluster / PSU", ["CLUSTER", "PSU", "STRATA", "SAMPLEWEIGHT"])

print("\n--- Duplicate PUD_ID examples (if any) ---")
dups = df[df["PUD_ID"].duplicated(keep=False)].sort_values("PUD_ID")
if len(dups):
    _show = [c for c in ["PUD_ID", "CLUSTER", "HH_LINE", "REGION", "AREA", "SEX"] if c in dups.columns]
    display(dups[_show].head(25))
else:
    print("none")


## 4. Harmonized codebook slots (Tanzania 2024)

File: `TANZANIA_VACS_2024_PUD.dta` under `data/raw/Tanzania 2024/`. For STRATA / CLUSTER / SAMPLEWEIGHT, see `TANZANIA_VACS_2024_DataUserGuide.pdf`.

Plain tab-separated block (select inside the fence, paste into Excel):

```
slot	pud_file	variables	type_and_width	notes
Respondent ID	TANZANIA_VACS_2024_PUD.dta	PUD_ID	str; 44-47 char length	11353 distinct / 11414 rows; 61 duplicate PUD_ID
Household ID	TANZANIA_VACS_2024_PUD.dta	CLUSTER + last underscore field of PUD_ID	str + str; CLUSTER 7 chars; segment 1-2 chars	Penultimate PUD_ID field equals CLUSTER all rows; 61 dup HH_KEY; no hh column
Geo level 1	TANZANIA_VACS_2024_PUD.dta	REGION	str; 4-16 char length	31 regions
Geo level 2	TANZANIA_VACS_2024_PUD.dta	AREA (optional REGI_C)	AREA str 5 chars (URBAN/RURAL); REGI_C float 1-31	
Cluster	TANZANIA_VACS_2024_PUD.dta	CLUSTER; PSU	CLUSTER str; 7 chars; style NNN-NNN (e.g. 133-133); PSU numeric (101–931)	svy: cluster=CLUSTER, strata=STRATA, weight=SAMPLEWEIGHT
Interview date	(not in PUD)	—	—	Fieldwork Mar-Jun 2024 per guide; no person-level date in dta
```

Excel note (Household ID): Type str + str; CLUSTER is 7 characters; line within cluster is 1–2 characters (last underscore-delimited field of PUD_ID; often numeric digits only).

Notebook: `HH_LINE` = last PUD_ID segment; `HH_KEY` = CLUSTER + "_" + HH_LINE.
